# PlaceMux AI Trust Sign-off: Proctoring FP Reduction

This notebook explores the synthetic proctoring session data, computes the v0 baseline metrics, and trains a RandomForest model to reduce False Positives.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

from src.data_loader import load_and_validate_data
from src.baseline import compute_baseline_metrics
from src.model import train_and_evaluate
from src.explainer import explain_prediction

# Load Data
df = load_and_validate_data('data/flagged_sessions.csv')
df.head()

Loaded 400 valid sessions. Deduplicated 5 rows.


,session_id,student_id,assessment_id,tab_switch_count,face_count_anomalies,copy_paste_events,time_per_question_zscore,network_latency_flag,webcam_dropout_seconds,ground_truth_label,scenario,flagged_by_v0,timestamp
0,62f7f85e-329f-4f52-8816-5a7d30353bcf,stu_0164,asm_002,2.0,0.0,1.0,-0.555940,0.0,2.0,NaN,normal_clean,0.0,2026-07-03 21:29:23.140937
1,ebce093e-5479-4b8c-85df-a5524d21ad3c,stu_0007,asm_005,2.0,0.0,1.0,0.159451,0.0,2.0,NaN,normal_clean,0.0,2026-07-03 21:30:23.140937
2,bdd35671-8974-4461-99da-10d7cd23720e,stu_0063,asm_004,1.0,0.0,0.0,-0.290439,0.0,3.0,NaN,normal_clean,0.0,2026-07-03 21:31:23.140937
3,19db0a91-a7ec-491a-be0e-b8c6e45e958f,stu_0036,asm_002,1.0,0.0,1.0,-0.262585,0.0,0.0,NaN,normal_clean,0.0,2026-07-03 21:32:23.140937
4,8af063ca-496b-4b7b-8e39-d313b3409a05,stu_0174,asm_009,0.0,0.0,0.0,-0.506416,0.0,2.0,NaN,normal_clean,0.0,2026-07-03 21:33:23.140937


## 1. Baseline Performance
Evaluating the `flagged_by_v0` rule.

In [2]:
baseline_metrics = compute_baseline_metrics(df)
print("Baseline FPR:", baseline_metrics['fpr'])

--- v0 Baseline Metrics ---
Evaluated on 100 labeled sessions
Precision: 0.3800
Recall: 1.0000
False Positive Rate (FPR): 1.0000
Absolute False Positives: 62
Baseline FPR: 1.0


## 2. Train Model and Reduce FPR
Train a Random Forest classifier, tune the threshold to maintain recall >= 0.8, and evaluate on the held-out test split.

In [3]:
results = train_and_evaluate(df, output_model_path='src/models/proctor_model.pkl')
print("Model FPR:", results['metrics']['model']['fpr'])

Chosen threshold: 0.10 (Val FPR: 0.0000)

--- Test Set Evaluation ---
Baseline -> FPR: 1.0000, Recall: 1.0000, FPs: 13
Model    -> FPR: 0.0000, Recall: 1.0000, FPs: 0
[SUCCESS] FPR reduced by 100.0%

--- Segment Breakdown (Test Set) ---
fp_network: Baseline flagged 6 -> Model cleared 6/6
fp_copypaste: Baseline flagged 7 -> Model cleared 7/7

Model saved to src/models/proctor_model.pkl
Model FPR: 0.0


## 3. Explainability and Edge Cases
Testing plain-English reasoning for a known False Positive pattern (e.g. high tab switch due to network latency).

In [4]:
# Get a known FP session that was flagged by v0
fp_candidates = df[(df["scenario"].isin(["fp_network", "fp_cat", "fp_copypaste"])) & (df["flagged_by_v0"] == 1)]
fp_session = fp_candidates.iloc[0].to_dict()

explanation = explain_prediction(fp_session, model_path='src/models/proctor_model.pkl')
print(f"v0 Flag: {fp_session['flagged_by_v0']}")
print(f"Model Verdict: {explanation['verdict']}")
print(f"Reason: {explanation['reason']}")

v0 Flag: 1.0
Model Verdict: cleared
Reason: Cleared: high tab-switch count (5.0) but network_issue_derived=1 — consistent with known connectivity FP pattern; model confidence 1.00 that this is NOT a violation.
